In [ ]:
# Create SparkSession

import os
import sys
import findspark

# 1. The correct Java 11 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

# 2. The Spark path
os.environ["SPARK_HOME"] = "/home/ubuntu/.local/lib/python3.10/site-packages/pyspark"

# 3. Force PySpark to use Jupyter's Python
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 4. Initialize findspark
findspark.init()

from pyspark.sql import SparkSession

# 5. Boot the Spark Session
spark = SparkSession.builder \
    .appName("NYCTaxi_Cluster_Analysis") \
    .getOrCreate()

print("Spark Session initialized successfully!")

In [ ]:
# Need to standardise the downloaded dataset 2020-2025, so aggregation can be done easily

import subprocess
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType, LongType

# Ask HDFS for a list of every raw file we have downloaded
cmd = "hdfs dfs -ls /data/nyc-taxi/*.parquet | awk '{print $8}'"
result = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, text=True)
file_paths = [line.strip() for line in result.stdout.split('\n') if line.strip().endswith('.parquet')]

print(f"Found {len(file_paths)} raw files. Starting standardization...")

clean_hdfs_path = "hdfs://group-32-master:9000/data/nyc-taxi-cleaned/"

# Process each file individually to bypass merge conflicts
for file_path in file_paths:
    # Adding hdfs:// prefix to the paths from the command line
    full_path = f"hdfs://group-32-master:9000{file_path}"
    
    try:
        # Read the single file (Spark infers this perfectly)
        temp_df = spark.read.parquet(full_path)
        
        # Manually cast every column to a strict, unified type
        clean_df = temp_df \
            .withColumn("passenger_count", col("passenger_count").cast(DoubleType())) \
            .withColumn("PULocationID", col("PULocationID").cast(LongType())) \
            .withColumn("DOLocationID", col("DOLocationID").cast(LongType())) \
            .withColumn("fare_amount", col("fare_amount").cast(DoubleType()))
            
        # Select ONLY the columns we need for the analysis
        final_df = clean_df.select(
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "passenger_count",
            "PULocationID",
            "DOLocationID",
            "fare_amount"
        )
        
        # Append this perfectly clean data to our new directory
        final_df.write.mode("append").parquet(clean_hdfs_path)
        print(f"Successfully cleaned & saved: {file_path.split('/')[-1]}")
        
    except Exception as e:
        print(f"Skipping {file_path} due to error: {e}")

print("\nData standardization complete! You now have a flawless dataset.")

In [ ]:
# PERFORM ANALYSIS

from pyspark.sql.functions import col, hour, unix_timestamp, avg, round

# 1. Load from your NEW, manually standardized folder
clean_data_path = "hdfs://group-32-master:9000/data/nyc-taxi-cleaned/*.parquet"
print("Loading clean dataset...")
df = spark.read.parquet(clean_data_path)

print(f"Total rows in clean dataset: {df.count()}")

# 2. Filter out invalid records
cleaned_df = df.filter(
    (col("fare_amount") > 0) & 
    (col("passenger_count") > 0) & 
    (col("PULocationID").isNotNull()) &
    (col("DOLocationID").isNotNull())
)

# 3. Transform the Data
transformed_df = cleaned_df \
    .withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
    .withColumn("duration_minutes", 
                (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60) \
    .filter(col("duration_minutes") > 0)

# 4. Aggregate
analysis_df = transformed_df.groupBy("PULocationID", "pickup_hour") \
    .agg(
        round(avg("duration_minutes"), 2).alias("avg_trip_duration"),
        round(avg("fare_amount"), 2).alias("avg_fare_amount")
    ).orderBy("PULocationID", "pickup_hour")

print("Aggregation complete. Showing top 20 results:")
analysis_df.show(20)

In [ ]:
lookup_path = "hdfs://group-32-master:9000/data/taxi_zone_lookup.csv"
lookup_df = spark.read.option("header", "true").option("inferSchema", "true").csv(lookup_path)

# Verify it loaded correctly
lookup_df.show(5)

In [ ]:
# Join the Zone Names
# We do a 'left' join matching the PULocationID to the LocationID in the lookup table
final_df = analysis_df.join(lookup_df, analysis_df.PULocationID == lookup_df.LocationID, "left")

# 5. Select, rename, and order the final columns
final_result = final_df.select(
    col("Zone").alias("pickup_zone"), # This gives us the string name instead of the ID
    "pickup_hour",
    "avg_trip_duration",
    "avg_fare_amount"
).orderBy("pickup_zone", "pickup_hour")

print("Final Output preview:")
final_result.show(20, truncate=False)

In [ ]:
# Save to CSV in HDFS
# coalesce(1) ensures the output is a single CSV file rather than split across multiple partitions
output_csv_path = "hdfs://group-32-master:9000/data/final_taxi_analysis.csv"

final_result.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(output_csv_path)

print(f"Analysis successfully saved to HDFS at: {output_csv_path}")